Resources for analysis:

* https://ucdavis-bioinformatics-training.github.io/2022-June-RNA-Seq-Analysis/data_analysis/DE_Analysis_mm_with_quizzes_fixed
* https://ucdavis-bioinformatics-training.github.io/2022-June-RNA-Seq-Analysis/data_reduction/03-counts_mm
* https://github.com/ben-laufer/RNA-seq/blob/main/04-limma-voom.R
* https://raw.githubusercontent.com/ucdavis-bioinformatics-training/2020-mRNA_Seq_Workshop/master/data_analysis/enrichment_mm.Rmd
* https://bioconductor.org/packages/release/bioc/vignettes/biomaRt/inst/doc/accessing_ensembl.html#selecting-an-ensembl-biomart-database-and-dataset
* https://cran.r-project.org/web/packages/enrichR/vignettes/enrichR.html

# Set Library Path

In [ ]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

# Load Libraries

In [ ]:
library(enrichR)
library(dplyr)
library(openxlsx)
library(ggplot2)
library(readxl)

## Gene Ontology

In [ ]:
# List all DEG files in the current subdirectory
files <- list.files(path = file.path("05_DEGs"), pattern = "\\_results.csv$", full.names = TRUE)

# Initialize an empty vector to hold files with significant DEGs
valid_files <- c()

# Loop through each file
for (file in files) {
  
  # Read the current file
  df <- read.csv(file)

  # Filter to only significant DEGs
  significant_df <- df[df$adj.P.Val < 0.05, ]

  # Check the number of significant DEGs
  num_significant <- nrow(significant_df)

  # Print the number of significant DEGs
  print(paste("File:", file, "- Number of significant DEGs:", num_significant))

  # If there are significant DEGs, keep the file
  if (num_significant > 0) {
    valid_files <- c(valid_files, file)
  }
}

# Update the original files list to only include valid files
files <- valid_files

# View the valid files
print(files)

In [ ]:
# Loop through each file
for (file in files) {
  
  # Read the current file
  df <- read.csv(file)

  # Filter to only significant DEGs
  df <- df[df$adj.P.Val < 0.05, ]

  # Sort by adjusted p-value and take the top 500
  df <- df[order(df$adj.P.Val), ]
  df <- head(df, 500)
    
  # Perform the enrichR analysis on the gene list
  enrichr_results <- enrichr(df$X, c("GO_Biological_Process_2025",
                                                      "GO_Cellular_Component_2025",
                                                      "GO_Molecular_Function_2025",
                                                      "KEGG_2021_Human",
                                                      "Panther_2016",
                                                      "Reactome_2022",
                                                      "RNA-Seq_Disease_Gene_and_Drug_Signatures_from_GEO"))
  
  # Create a new Excel workbook for the current file
  wb <- createWorkbook()
  
  # Loop through each table in enrichr_results and save it as a separate sheet in the Excel workbook
  for (i in seq_along(enrichr_results)) {
    # Extract the data frame from the list
    enrichr_df <- enrichr_results[[i]]
    
    # Define the original sheet name
    original_sheet_name <- names(enrichr_results)[i]
    
    # Modify the sheet name if it's specifically "RNA-Seq_Disease_Gene_and_Drug_Signatures_from_GEO"
    sheet_name <- if (original_sheet_name == "RNA-Seq_Disease_Gene_and_Drug_Signatures_from_GEO") {
      "RNAseq_DiseaseGene_DrugSigs_GEO"
    } else {
      original_sheet_name
    }
    
    # Add the data frame as a new sheet in the Excel workbook
    addWorksheet(wb, sheet_name)
    writeData(wb, sheet = sheet_name, x = enrichr_df)
  }
    
  # Define the output file name
  output_filename <- paste0("06_DEG_GO/", tools::file_path_sans_ext(basename(file)), "_enrichr_results.xlsx")
  
  # Save the Excel workbook for this file
  saveWorkbook(wb, output_filename, overwrite = TRUE)
  
  # Print message indicating successful save
  cat("Enrichr results saved for:", file, "->", output_filename, "\n")
}

## Plot All Genes

In [ ]:
# Define functions to calculate -log10 of P-value and filter top 10 terms
calculate_neg_log_p <- function(df) {
  df %>%
    mutate(neg_log_p = ifelse(P.value > 0, -log10(P.value), Inf))
}

filter_top_10 <- function(df) {
  return(df %>% arrange(P.value) %>% head(10))
}

In [ ]:
# List all Enrichr Excel files
files <- list.files(path = "06_DEG_GO", pattern = "\\.xlsx$", full.names = TRUE, recursive = TRUE)

# Loop through each Excel file and make graphs
for (file in files) {
  # Extract the base filename without extension
  filename <- tools::file_path_sans_ext(basename(file))
  
  # Get the gene name from the directory path
  gene_name <- basename(dirname(file))
  
  # Extract comparison name from filename
  comparison_name <- gsub("_enrichr_results", "", filename)
  comparison_name <- gsub("_", " ", comparison_name)
  
  cat("\n=== ", gene_name, " ===\n")
  cat("Comparison:", comparison_name, "\n")
  
  # Read Enrichr results from Excel sheets
  enrichr_results <- list(
    GO_Biological_Process_2025 = read_excel(file, sheet = "GO_Biological_Process_2025"),
    GO_Cellular_Component_2025 = read_excel(file, sheet = "GO_Cellular_Component_2025"),
    GO_Molecular_Function_2025 = read_excel(file, sheet = "GO_Molecular_Function_2025")
  )
    
  # Process each GO category
  GO_BP <- enrichr_results$GO_Biological_Process_2025
  GO_CC <- enrichr_results$GO_Cellular_Component_2025
  GO_MF <- enrichr_results$GO_Molecular_Function_2025
  
  # Filter top 10 terms by p-value
  GO_BP <- GO_BP %>% arrange(P.value) %>% head(10)
  GO_CC <- GO_CC %>% arrange(P.value) %>% head(10)
  GO_MF <- GO_MF %>% arrange(P.value) %>% head(10)
  
  # Calculate -log10 p-values
  GO_BP$neg_log_p <- -log10(GO_BP$P.value)
  GO_CC$neg_log_p <- -log10(GO_CC$P.value)
  GO_MF$neg_log_p <- -log10(GO_MF$P.value)

  # Combine data frames for plotting
  combined_df <- rbind(
    mutate(GO_BP, Category = "Biological Process"),
    mutate(GO_CC, Category = "Cellular Component"),
    mutate(GO_MF, Category = "Molecular Function")
  )
    
  # Skip if no data
  if (nrow(combined_df) == 0) {
    cat("No Enrichr results for this comparison\n")
    next
  }
       
  # Plot
  pdf(paste0("06_DEG_GO/", comparison_name, "_enrichr_barplot.pdf"), height = 5, width = 12)

  p <- ggplot(combined_df, aes(x = neg_log_p, y = reorder(Term, neg_log_p), fill = Category)) +
    geom_col(position = position_dodge(width = 0.9)) +
    scale_fill_manual(values = c("Biological Process" = "#2B79B2", 
                                  "Cellular Component" = "#FA7C1A", 
                                  "Molecular Function" = "#3CA036")) +
    labs(x = "-log10(P-value)", y = NULL, 
         title = paste("Enrichr Results:", comparison_name)) +
    facet_grid(Category ~ ., scales = "free_y", space = "free_y") +
    theme_minimal() +
    theme(legend.position = "right",
          strip.text.y = element_blank(), 
          panel.grid.major = element_blank(),  
          panel.grid.minor = element_blank())

  print(p)
    
  dev.off()

  # Print results
  cat("Biological Process terms:", nrow(GO_BP), "\n")
  cat("Cellular Component terms:", nrow(GO_CC), "\n")
  cat("Molecular Function terms:", nrow(GO_MF), "\n")
  cat("Plot generated for:", comparison_name, "\n")
}

In [ ]:
# List all Enrichr Excel files
files <- list.files(path = "06_DEG_GO", pattern = "\\.xlsx$", full.names = TRUE, recursive = TRUE)

# Mapping of sheet names to database names
sheet_mapping <- c(
  "GO_Biological_Process_2025" = "GO_Biological_Process_2025",
  "GO_Cellular_Component_2025" = "GO_Cellular_Component_2025",
  "GO_Molecular_Function_2025" = "GO_Molecular_Function_2025",
  "KEGG_2021_Human" = "KEGG_2021_Human",
  "Panther_2016" = "Panther_2016",
  "Reactome_2022" = "Reactome_2022",
  "RNAseq_DiseaseGene_DrugSigs_GEO" = "RNAseq_DiseaseGene_DrugSigs_GEO"
)

# Loop through each Excel file
for (file in files) {
  # Get the gene name from the directory path
  gene_name <- basename(dirname(file))
  
  # Extract comparison name from filename
  comparison_name <- gsub("_enrichr_results", "", basename(file))
  comparison_name <- gsub("\\.xlsx$", "", comparison_name)
  comparison_name <- gsub("_", " ", comparison_name)
  
  cat("\n=== ", gene_name, " ===\n")
  cat("Comparison:", comparison_name, "\n")
  
  # Loop through each database sheet
  for (sheet_name in names(sheet_mapping)) {
    tryCatch({
      # Read the sheet
      db_terms <- read_excel(file, sheet = sheet_mapping[sheet_name])
      
      # Skip if no data
      if (nrow(db_terms) == 0) {
        cat("No data for", sheet_name, "\n")
        next
      }
      
      # Define the PDF filename
      pdf_filename <- paste0("06_DEG_GO/", comparison_name, "_", sheet_name, ".pdf")
      
      # Create the plot and save it
      pdf(pdf_filename, height = 7, width = 17)
      p <- plotEnrich(db_terms, showTerms = 25, numChar = 75, y = "Count", orderBy = "P.value") +
        ggtitle(paste(sheet_name, ":", comparison_name))
      
      print(p)
      dev.off()
      
      # Print success message
      cat("Plot saved:", sheet_name, "->", pdf_filename, "\n")
      
    }, error = function(e) {
      # Handle any read errors
      cat("Error reading", sheet_name, ":", e$message, "\n")
    })
  }
}